In [ ]:
from uq_physicell.model_analysis import ModelAnalysisContext, run_simulations

# Run a single simulation to generate observational data
db_path = "Ex7_ObsData.db"  # Path to the database file
model_config = {"ini_path": "uq_pc_struc.ini", "struc_name": "Model_struc"} # Example model configuration
sampler = 'Custom'  # Example sampler
params_info = { # Example parameters information
    'mac_phag_rate_infected': {'ref_value': 1.0, 'perturbation': None},
    'mac_motility_bias': {'ref_value': 0.15, 'perturbation': None},
    'epi2infected_sat': {'ref_value': 0.1, 'perturbation': None},
    'epi2infected_hfm': {'ref_value': 0.4, 'perturbation': None}
}  
qoi_funcs = {
    'epithelial': "lambda df: len(df[df['cell_type'] == 'epithelial'])",
    'epithelial_infected': "lambda df: len(df[df['cell_type'] == 'epithelial_infected'])"    
}
# Create Model Analysis Context
context = ModelAnalysisContext(db_path, model_config, sampler, params_info, qoi_funcs, num_workers=8)
# Sample parameter sets and run simulations
context.dic_samples = {0: {'mac_phag_rate_infected': 1.0, 'mac_motility_bias': 0.15, 'epi2infected_sat': 0.1, 'epi2infected_hfm': 0.4}}
run_simulations(context)

Inserting parameter: mac_phag_rate_infected with properties: {'ref_value': 1.0, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting parameter: mac_motility_bias with properties: {'ref_value': 0.15, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting parameter: epi2infected_sat with properties: {'ref_value': 0.1, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting parameter: epi2infected_hfm with properties: {'ref_value': 0.4, 'perturbation': None, 'lower_bound': None, 'upper_bound': None}
Inserting {'epithelial': "lambda df: len(df[df['cell_type'] == 'epithelial'])", 'epithelial_infected': "lambda df: len(df[df['cell_type'] == 'epithelial_infected'])"} QoIs into the database
Simulations completed and results stored in the database: Ex7_ObsData.db.


In [3]:
import pandas as pd
from uq_physicell.database.ma_db import load_output
# Load the simulation results and extract cell counts over time
df_qois_data = load_output(db_path)
df_all_data = pd.DataFrame()
for index, row in df_qois_data.iterrows():
    df_temp = row['Data']
    df_all_data = pd.concat([df_all_data, df_temp], ignore_index=True)

# Aggregate with multiple summary statistics
df_aggregated = df_all_data.groupby('time').agg({
    'epithelial': ['mean', 'std', 'min', 'max', 'count'],
    'epithelial_infected': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()

# Flatten column names for easier access
df_aggregated.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                          for col in df_aggregated.columns.values]

time = df_aggregated['time'].values
live = df_aggregated['epithelial_mean'].values
dead = df_aggregated['epithelial_infected_mean'].values
df_aggregated.rename(columns={'time': 'Time', 'epithelial_mean': 'Healthy Epithelial Cells', 'epithelial_infected_mean': 'Infected Epithelial Cells'}, inplace=True)
display(df_aggregated)
df_aggregated.to_csv("ex7_ObsData.csv", index=False)

,Time,Healthy Epithelial Cells,epithelial_std,epithelial_min,epithelial_max,epithelial_count,Infected Epithelial Cells,epithelial_infected_std,epithelial_infected_min,epithelial_infected_max,epithelial_infected_count
0,0.0,250.0,0.000000,250,250,5,20.0,0.000000,20,20,5
1,360.0,244.4,0.547723,244,245,5,35.4,3.577709,31,38,5
2,720.0,221.6,5.272571,213,225,5,70.6,4.669047,65,78,5
3,1080.0,124.2,6.572671,117,135,5,195.4,11.696153,175,205,5
4,1440.0,74.0,10.099505,68,92,5,296.4,16.303374,270,315,5
5,1800.0,44.6,15.805062,24,56,5,413.4,21.466253,399,447,5
6,2160.0,11.8,7.661593,0,17,5,561.4,22.864820,546,597,5
7,2520.0,0.0,0.000000,0,0,5,747.4,13.145341,724,754,5
8,2880.0,0.0,0.000000,0,0,5,955.8,33.677886,902,978,5
